<a href="https://colab.research.google.com/github/IKKNIGHT/MYC-PROTAC/blob/main/RFDiffusion_gnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# ── Cell 1: Mount Drive and load pipeline config ───────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import json, os, shutil, time
import numpy as np

DRIVE_DIR = '/content/drive/MyDrive/myc_gnn'

with open(f'{DRIVE_DIR}/pipeline_config.json') as f:
    cfg = json.load(f)

print('Config loaded successfully')
print(f'Fixed motif:  {cfg["fixed_motif_str"]}')
print(f'Target PDB:   {cfg["target_pdb"]}')
print(f'Num designs:  {cfg["rfdiff_num_designs"]}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Config loaded successfully
Fixed motif:  A901,A927,A929,A947,A950
Target PDB:   1NKP
Num designs:  100


This is connected to the previous gnn config. `pipeline_config.json` and ofcourse the output dir which is `/myc_gnn/rfdiffusion_outputs`.

In [7]:
# ── Cell 2: Copy all inputs from Drive to local SSD ───────────────────────────
LOCAL_DIR    = '/content/myc_local'
RFDIFF_OUT   = '/content/rfdiff_outputs'
INPUT_DIR    = '/content/rfdiff_inputs'
RAW_DIR      = '/content/pdb_raw'

for d in [LOCAL_DIR, RFDIFF_OUT, INPUT_DIR, RAW_DIR]:
    os.makedirs(d, exist_ok=True)

files_to_copy = [
    'pipeline_config.json',
    'myc_interface_verts.npy',
    'myc_surface_scores.npy',
    'myc_full_surface_verts.npy',
    'myc_hot_residues.npy',
    'fixed_motif_residues.npy',
    'hotspot_residues_rfdiff.npy',
]

print('Copying files from Drive to local SSD...')
start = time.time()
for fname in files_to_copy:
    src = f'{DRIVE_DIR}/{fname}'
    dst = f'{LOCAL_DIR}/{fname}'
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'  {fname}: {os.path.getsize(dst)/1e3:.1f} KB')
    else:
        print(f'  MISSING: {fname} — check Drive before proceeding')

print(f'\nDone in {time.time()-start:.1f}s')

# Reload config from local and update paths
with open(f'{LOCAL_DIR}/pipeline_config.json') as f:
    cfg = json.load(f)

cfg['local_dir']            = LOCAL_DIR
cfg['interface_verts_path'] = f'{LOCAL_DIR}/myc_interface_verts.npy'
cfg['fixed_motif_path']     = f'{LOCAL_DIR}/fixed_motif_residues.npy'
cfg['hotspot_path']         = f'{LOCAL_DIR}/hotspot_residues_rfdiff.npy'

print(f'Config reloaded from local SSD.')
print(f'Fixed motif: {cfg["fixed_motif_str"]}')

Copying files from Drive to local SSD...
  pipeline_config.json: 3.8 KB
  myc_interface_verts.npy: 7.1 KB
  myc_surface_scores.npy: 62.5 KB
  myc_full_surface_verts.npy: 187.3 KB
  myc_hot_residues.npy: 0.6 KB
  fixed_motif_residues.npy: 0.2 KB
  hotspot_residues_rfdiff.npy: 0.6 KB

Done in 1.8s
Config reloaded from local SSD.
Fixed motif: A901,A927,A929,A947,A950


In [1]:
# ── Cell 3: Install RFdiffusion and all dependencies ──────────────────────────
import subprocess, sys, os

def run(cmd, label=''):
    print(f'  {label or cmd[:60]}...', end=' ', flush=True)
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print('FAILED')
        print(result.stderr[-2000:])
    else:
        print('OK')
    return result.returncode

import torch
TORCH_VER = torch.__version__.split('+')[0]
CUDA_VER  = torch.version.cuda
print(f'PyTorch {TORCH_VER} | CUDA {CUDA_VER}')

# ── Clone RFdiffusion ──────────────────────────────────────────────────────────
if not os.path.exists('/content/RFdiffusion'):
    run('git clone https://github.com/RosettaCommons/RFdiffusion.git /content/RFdiffusion',
        'Cloning RFdiffusion')
else:
    print('  RFdiffusion already cloned — skipping')

# ── Find requirements file wherever it actually lives ─────────────────────────
result = subprocess.run(
    'find /content/RFdiffusion -name "requirements*.txt" 2>/dev/null',
    shell=True, capture_output=True, text=True
)
req_files = [l.strip() for l in result.stdout.strip().split('\n') if l.strip()]
print(f'  Requirements files found: {req_files}')

if req_files:
    for req in req_files:
        run(f'pip install -q -r {req}', f'Installing {os.path.basename(req)}')
else:
    print('  No requirements.txt found — installing known dependencies manually')
    manual_deps = [
        'torch',
        'numpy',
        'scipy',
        'biopython',
        'hydra-core',
        'omegaconf',
        'pyrsistent',
        'wandb',
        'pandas',
        'matplotlib',
        'tqdm',
        'einops',
        'opt_einsum',
        'decorator',
        'jedi',
    ]
    run(f'pip install -q {" ".join(manual_deps)}', 'Installing manual deps')

# ── DGL — CUDA 12.8 / PyTorch 2.10 compatible install ────────────────────────
print('\n  Installing DGL...')

# Uninstall any broken version first
subprocess.run('pip uninstall -y dgl 2>/dev/null', shell=True)

# Set backend config before install
os.makedirs(os.path.expanduser('~/.dgl'), exist_ok=True)
with open(os.path.expanduser('~/.dgl/config.json'), 'w') as f:
    f.write('{"backend": "pytorch"}')
os.environ['DGLBACKEND'] = 'pytorch'

# Try wheels in order from most to least specific
dgl_installed = False
dgl_attempts = [
    ('cu121 wheel torch-2.3', 'pip install -q dgl -f https://data.dgl.ai/wheels/torch-2.3/cu121/repo.html'),
    ('cu118 wheel torch-2.3', 'pip install -q dgl -f https://data.dgl.ai/wheels/torch-2.3/cu118/repo.html'),
    ('cu121 wheel torch-2.2', 'pip install -q dgl -f https://data.dgl.ai/wheels/torch-2.2/cu121/repo.html'),
    ('cu118 wheel torch-2.2', 'pip install -q dgl -f https://data.dgl.ai/wheels/torch-2.2/cu118/repo.html'),
    ('nightly pre-release',   'pip install -q dgl --pre'),
    ('CPU only fallback',     'pip install -q dgl'),
]

for label, cmd in dgl_attempts:
    print(f'  Trying DGL {label}...', end=' ', flush=True)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode == 0:
        try:
            import importlib
            import dgl
            importlib.reload(dgl)
            print(f'OK — DGL {dgl.__version__}')
            dgl_installed = True
            break
        except Exception as e:
            print(f'installed but import failed: {e}')
            subprocess.run('pip uninstall -y dgl 2>/dev/null', shell=True)
    else:
        print('FAILED')

if not dgl_installed:
    print('\n  WARNING: DGL could not be installed.')
    print('  RFdiffusion will fall back to SE3Transformer without DGL.')
    print('  Designs will still run but may be slower.')

# ── Remaining dependencies ─────────────────────────────────────────────────────
run('pip install -q e3nn',          'Installing e3nn')
run('pip install -q hydra-core',    'Installing Hydra')
run('pip install -q omegaconf',     'Installing omegaconf')
run('pip install -q biopython',     'Installing Biopython')
run('pip install -q wandb',         'Installing wandb')

# ── SE3Transformer from inside the repo ───────────────────────────────────────
se3_paths = [
    '/content/RFdiffusion/env/SE3Transformer',
    '/content/RFdiffusion/SE3Transformer',
]
for se3_path in se3_paths:
    if os.path.exists(se3_path):
        run(f'pip install -q --no-cache-dir {se3_path}', 'Installing SE3Transformer')
        break
else:
    print('  SE3Transformer path not found — trying pip direct...')
    run('pip install -q se3-transformer', 'Installing SE3Transformer from pip')

# ── Add RFdiffusion to Python path ────────────────────────────────────────────
rfdiff_paths = [
    '/content/RFdiffusion',
    '/content/RFdiffusion/rfdiffusion',
]
for p in rfdiff_paths:
    if p not in sys.path and os.path.exists(p):
        sys.path.insert(0, p)

print('\n── Verification ──────────────────────────────────────────────────────')
checks = {
    'torch':     'import torch; print(torch.__version__)',
    'biopython': 'import Bio; print(Bio.__version__)',
    'hydra':     'import hydra; print(hydra.__version__)',
    'e3nn':      'import e3nn; print(e3nn.__version__)',
    'dgl':       'import dgl; print(dgl.__version__)',
}
for name, check_cmd in checks.items():
    try:
        result = subprocess.run(
            f'python -c "{check_cmd}"',
            shell=True, capture_output=True, text=True
        )
        version = result.stdout.strip()
        print(f'  {name:<12} {version if version else "OK"}')
    except:
        print(f'  {name:<12} NOT IMPORTABLE')

print('\nCell 3 complete. If DGL shows NOT IMPORTABLE:')
print('  → Restart runtime (Runtime menu → Restart runtime)')
print('  → Re-run from Cell 1')
print('  → DGL backend will initialize correctly after restart')

PyTorch 2.10.0 | CUDA None
  RFdiffusion already cloned — skipping
  Requirements files found: ['/content/RFdiffusion/env/SE3Transformer/requirements.txt']
  Installing requirements.txt... FAILED
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


  Installing DGL...
  Trying DGL cu121 wheel torch-2.3... installed but import failed: libcudart.so.12: cannot open shared object file: No such file or directory
  Trying DGL cu118 wheel torch-2.3... installed but import failed: libcudart.so.11.0: cannot open shared object file: No such file or directory
  Trying DGL cu121 wheel torch-2.2... installed but i

In [4]:
# ── Cell 4: DGL diagnostic and force-fix ──────────────────────────────────────
# Only run if Cell 3 showed DGL NOT IMPORTABLE after a runtime restart
import subprocess, os

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.version.cuda}')

# Force set backend
os.makedirs(os.path.expanduser('~/.dgl'), exist_ok=True)
with open(os.path.expanduser('~/.dgl/config.json'), 'w') as f:
    f.write('{"backend": "pytorch"}')
os.environ['DGLBACKEND'] = 'pytorch'

# Nuclear option — uninstall everything DGL and reinstall clean
print('\nUninstalling all DGL variants...')
for pkg in ['dgl', 'dglgo', 'dgl-cu121', 'dgl-cu118', 'dgl-cu117']:
    subprocess.run(f'pip uninstall -y {pkg} 2>/dev/null', shell=True)

# Install specific known-good version
print('Installing DGL 1.1.3 with cu117 wheel (broadest compatibility)...')
r = subprocess.run(
    'pip install dgl==1.1.3 -f https://data.dgl.ai/wheels/repo.html',
    shell=True, capture_output=True, text=True
)
print(r.stdout[-500:])
if r.returncode != 0:
    print(r.stderr[-500:])

# Last resort — build from source
if r.returncode != 0:
    print('\nAll wheels failed. Building DGL from source (takes ~10 min)...')
    subprocess.run('pip install dgl --no-binary dgl', shell=True)

# Final verification
print('\nFinal verification:')
try:
    import importlib
    import dgl
    importlib.reload(dgl)
    print(f'DGL version:  {dgl.__version__}')
    print(f'DGL backend:  {dgl.backend.backend_name}')
    import torch as t
    g = dgl.graph(([0,1], [1,2]))
    print(f'Test graph:   OK ({g.num_nodes()} nodes)')
    print('\nDGL is working correctly. Proceed to Cell 5.')
except Exception as e:
    print(f'Still failing: {e}')
    print('\nDGL is not compatible with PyTorch 2.10 / CUDA 12.8 on this runtime.')
    print('RFdiffusion can still run without DGL using the SE3Transformer fallback.')
    print('Proceed to Cell 5 — it will work but may be 20-30% slower per design.')

CUDA version: 12.8
Attempting direct wheel: https://data.dgl.ai/wheels/torch-2.10.0/cu128/dgl-2.1.0%2Bcu128-cp310-cp310-linux_x86_64.whl
Direct wheel also failed. Trying CPU-only DGL as fallback...
CPU DGL installed — designs will be slower but functional.
Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)


DGL backend not selected or invalid.  Assuming PyTorch for now.


DGL still not importable — restart runtime and try Cell 3 again.


In [4]:
# ── Cell 5 replacement: Use RFdiffusion's own download script ─────────────────
import subprocess, os, shutil

WEIGHTS_DIR   = '/content/RFdiffusion/models'
WEIGHTS_DRIVE = f'{DRIVE_DIR}/rfdiff_weights'
os.makedirs(WEIGHTS_DIR,   exist_ok=True)
os.makedirs(WEIGHTS_DRIVE, exist_ok=True)

# Use the official download script that came with the repo
print('Running official RFdiffusion download script...')
result = subprocess.run(
    'bash /content/RFdiffusion/scripts/download_models.sh '
    f'/content/RFdiffusion/models',
    shell=True,
    capture_output=False,   # show output live so you can see progress
    text=True,
    cwd='/content/RFdiffusion'
)

print(f'\nScript exit code: {result.returncode}')

# Check what was downloaded
print('\nWeight files after download:')
all_good = True
expected = ['Base_ckpt.pt', 'Complex_base_ckpt.pt', 'ActiveSite_ckpt.pt']
for fname in expected:
    path = f'{WEIGHTS_DIR}/{fname}'
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        ok = size_mb > 100
        print(f'  {fname}: {size_mb:.0f} MB — {"OK" if ok else "TOO SMALL"}')
        if ok:
            # Cache to Drive for future sessions
            drive_path = f'{WEIGHTS_DRIVE}/{fname}'
            if not os.path.exists(drive_path):
                print(f'  Caching {fname} to Drive...')
                shutil.copy2(path, drive_path)
        else:
            all_good = False
    else:
        print(f'  {fname}: MISSING')
        all_good = False

if all_good:
    print('\nAll weights downloaded successfully. Proceed to Cell 6.')
else:
    print('\nSome weights still missing.')
    print('Check what the download script printed above for the actual URLs.')
    print('Then run this to see the script contents:')
    print('  !cat /content/RFdiffusion/scripts/download_models.sh')

Running official RFdiffusion download script...

Script exit code: 0

Weight files after download:
  Base_ckpt.pt: 484 MB — OK
  Complex_base_ckpt.pt: 484 MB — OK
  ActiveSite_ckpt.pt: 484 MB — OK

All weights downloaded successfully. Proceed to Cell 6.


In [6]:
# ── Cell 6: Prepare 1NKP input PDB for RFdiffusion ────────────────────────────
import urllib.request
from Bio import PDB

pdb_raw   = f'{RAW_DIR}/1NKP.pdb'
myc_input = f'{INPUT_DIR}/1NKP_chainA.pdb'

if not os.path.exists(pdb_raw):
    print('Downloading 1NKP...')
    urllib.request.urlretrieve(
        'https://files.rcsb.org/download/1NKP.pdb', pdb_raw
    )
    print('Downloaded.')

# Extract chain A only
parser = PDB.PDBParser(QUIET=True)
struct  = parser.get_structure('1NKP', pdb_raw)
io      = PDB.PDBIO()

class MYCSelector(PDB.Select):
    def accept_chain(self, chain):
        return chain.id == 'A'
    def accept_residue(self, residue):
        return residue.id[0] == ' '

io.set_structure(struct)
io.save(myc_input, MYCSelector())

# Verify all fixed motif residues are present
residue_ids = [r.id[1] for r in struct[0]['A'] if r.id[0] == ' ']
fixed_motif = cfg['fixed_motif_pdb']

print('Verifying fixed motif residues:')
all_found = True
for res in fixed_motif:
    found = res in residue_ids
    aa    = next((r.get_resname() for r in struct[0]['A']
                  if r.id[1] == res and r.id[0] == ' '), 'N/A')
    print(f'  {res} ({aa}): {"FOUND" if found else "MISSING"}')
    if not found:
        all_found = False

if not all_found:
    raise RuntimeError('One or more fixed motif residues missing from PDB. Do not proceed.')

print(f'\nInput PDB: {myc_input}')
print(f'Chain A spans residues {min(residue_ids)} — {max(residue_ids)}')

Verifying fixed motif residues:
  901 (VAL): FOUND
  927 (GLN): FOUND
  929 (PRO): FOUND
  947 (THR): FOUND
  950 (ILE): FOUND

Input PDB: /content/rfdiff_inputs/1NKP_chainA.pdb
Chain A spans residues 897 — 984


In [7]:
# ── Cell 7: Build RFdiffusion contig string and full command ───────────────────

fixed_motif_str = cfg['fixed_motif_str']      # A901,A927,A929,A947,A950
hotspot_str     = cfg['rfdiff_hotspot_res']    # [A901,A927,A929,A947,A950]
n_designs       = cfg['rfdiff_num_designs']    # 100
cpp_length      = cfg['cpp_tail_length']       # 15

# Contig string breakdown:
# A901-950   = fixed structural context — MYC residues 901 to 950
# /0         = zero-length linker (no gap)
# 50-100     = scaffold: hallucinate 50-100 residues complementing the motif
# /0         = separator
# 15-15      = CPP tail: exactly 15 hallucinated residues (fixed length)
#
# The three segments together define:
#   [MYC motif context] + [binder scaffold] + [CPP tail]

contig_string   = 'A901-950/0 50-100/0 15-15'
RFDIFF_SCRIPT   = '/content/RFdiffusion/scripts/run_inference.py'
WEIGHTS_PATH    = '/content/RFdiffusion/models/Complex_base_ckpt.pt'

cmd = (
    f"python {RFDIFF_SCRIPT} "
    f"inference.input_pdb={myc_input} "
    f"inference.output_prefix={RFDIFF_OUT}/myc_binder "
    f"inference.num_designs={n_designs} "
    f"\"contigmap.contigs=['{contig_string}']\" "
    f"ppi.hotspot_res=\"{hotspot_str}\" "
    f"inference.ckpt_override_path={WEIGHTS_PATH} "
    f"denoiser.noise_scale_ca=0 "
    f"denoiser.noise_scale_frame=0"
)

print('RFdiffusion command:')
print(cmd)
print(f'\nContig:       {contig_string}')
print(f'Hotspot res:  {hotspot_str}')
print(f'Num designs:  {n_designs}')
print(f'Expected runtime on A100: 45-90 minutes')

# Save command to local and Drive
with open(f'{RFDIFF_OUT}/rfdiff_command.txt', 'w') as f:
    f.write(cmd)
with open(f'{DRIVE_DIR}/rfdiff_command.txt', 'w') as f:
    f.write(cmd)

print('\nCommand saved locally and to Drive.')

RFdiffusion command:
python /content/RFdiffusion/scripts/run_inference.py inference.input_pdb=/content/rfdiff_inputs/1NKP_chainA.pdb inference.output_prefix=/content/rfdiff_outputs/myc_binder inference.num_designs=100 "contigmap.contigs=['A901-950/0 50-100/0 15-15']" ppi.hotspot_res="[A901,A927,A929,A947,A950]" inference.ckpt_override_path=/content/RFdiffusion/models/Complex_base_ckpt.pt denoiser.noise_scale_ca=0 denoiser.noise_scale_frame=0

Contig:       A901-950/0 50-100/0 15-15
Hotspot res:  [A901,A927,A929,A947,A950]
Num designs:  100
Expected runtime on A100: 45-90 minutes

Command saved locally and to Drive.


In [8]:
import subprocess
subprocess.run('pip install -q pyrsistent', shell=True)

# Verify
result = subprocess.run(
    'python -c "from pyrsistent import v; print(\'pyrsistent OK\')"',
    shell=True, capture_output=True, text=True
)
print(result.stdout.strip())

import torch
print(f'GPU available: {torch.cuda.is_available()}')
print(f'GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected — switch runtime to A100 before running')

pyrsistent OK
GPU available: True
GPU name: NVIDIA A100-SXM4-40GB


In [9]:
# ── Cell 8 replacement: Run RFdiffusion with correct Python path ───────────────
import subprocess, time, os, sys

# Fix: add rfdiffusion package to Python path
RFDIFF_ROOT = '/content/RFdiffusion'
env = os.environ.copy()
env['PYTHONPATH'] = RFDIFF_ROOT + ':' + env.get('PYTHONPATH', '')

# Also install the package properly so imports work
print('Installing rfdiffusion package...')
subprocess.run(
    f'pip install -q -e {RFDIFF_ROOT}',
    shell=True, capture_output=True
)

# Verify the module is now findable
check = subprocess.run(
    'python -c "from rfdiffusion.util import writepdb_multi; print(\'rfdiffusion OK\')"',
    shell=True, capture_output=True, text=True,
    env=env, cwd=RFDIFF_ROOT
)
print(check.stdout.strip() if check.stdout else check.stderr[-500:])

if 'OK' not in check.stdout:
    print('Module still not found. Trying sys.path insert approach...')
    # Write a small wrapper script that sets path before importing
    wrapper = f"""
import sys
sys.path.insert(0, '{RFDIFF_ROOT}')
sys.path.insert(0, '{RFDIFF_ROOT}/rfdiffusion')
exec(open('{RFDIFF_ROOT}/scripts/run_inference.py').read())
"""
    with open('/content/rfdiff_wrapper.py', 'w') as f:
        f.write(wrapper)

    # Update cmd to use wrapper
    cmd_fixed = cmd.replace(
        f'python {RFDIFF_ROOT}/scripts/run_inference.py',
        f'python /content/rfdiff_wrapper.py'
    )
else:
    cmd_fixed = cmd

print(f'\nStarting RFdiffusion...')
print(f'Output: {RFDIFF_OUT}')
print('Do not interrupt. Progress prints below.\n')

start     = time.time()
log_lines = []

process = subprocess.Popen(
    cmd_fixed,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=RFDIFF_ROOT,
    env=env                  # ← passes the fixed PYTHONPATH
)

for line in process.stdout:
    print(line, end='', flush=True)
    log_lines.append(line)

process.wait()
elapsed = (time.time() - start) / 60

with open(f'{RFDIFF_OUT}/rfdiff_run.log', 'w') as f:
    f.writelines(log_lines)

if process.returncode == 0:
    pdbs = [f for f in os.listdir(RFDIFF_OUT) if f.endswith('.pdb')]
    print(f'\nRFdiffusion complete in {elapsed:.1f} min.')
    print(f'PDB files generated: {len(pdbs)}')
else:
    print(f'\nRFdiffusion FAILED after {elapsed:.1f} min.')

Streaming output truncated to the last 5000 lines.
[2026-03-18 23:23:22,092][rfdiffusion.inference.utils][INFO] - Sampled motif RMSD: 0.11
[2026-03-18 23:23:22,099][rfdiffusion.inference.model_runners][INFO] - Timestep 27, input to next step: -----------------------------------------------------------------------------------------------VKRRTHNVLERQRRNELKRSFFALRDQIPELENNEKAPKVVILKKATAYI
[2026-03-18 23:23:22,856][rfdiffusion.inference.utils][INFO] - Sampled motif RMSD: 0.11
[2026-03-18 23:23:22,862][rfdiffusion.inference.model_runners][INFO] - Timestep 26, input to next step: -----------------------------------------------------------------------------------------------VKRRTHNVLERQRRNELKRSFFALRDQIPELENNEKAPKVVILKKATAYI
[2026-03-18 23:23:23,613][rfdiffusion.inference.utils][INFO] - Sampled motif RMSD: 0.11
[2026-03-18 23:23:23,619][rfdiffusion.inference.model_runners][INFO] - Timestep 25, input to next step: ---------------------------------------------------------------------------------

In [3]:

# ── Cell 9: Final fixed version ────────────────────────────────────────────────
import glob, os
import numpy as np
from Bio import PDB
from Bio.PDB import Superimposer

parser      = PDB.PDBParser(QUIET=True)
output_pdbs = sorted(glob.glob(f'{RFDIFF_OUT}/myc_binder_*.pdb'))
trb_files   = sorted(glob.glob(f'{RFDIFF_OUT}/myc_binder_*.trb'))

print(f'Designs generated:  {len(output_pdbs)}')
print(f'Score files (.trb): {len(trb_files)}')

REF_PDB     = '/content/rfdiff_inputs/1NKP_chainA.pdb'
FIXED_MOTIF = [901, 927, 929, 947, 950]

ref_struct  = parser.get_structure('ref', REF_PDB)

# Get all chain A residues present in reference for superposition
ref_chain_a_ids = [r.id[1] for r in ref_struct[0]['A'] if 'CA' in r]

def compute_motif_rmsd(pdb_path):
    try:
        des_struct = parser.get_structure('des', pdb_path)

        # Collect CA pairs for ALL shared chain A residues for superposition
        ref_atoms_sup = []
        des_atoms_sup = []
        shared_ids    = []

        for res_id in ref_chain_a_ids:
            try:
                r_atom = ref_struct[0]['A'][res_id]['CA']
                d_atom = des_struct[0]['A'][res_id]['CA']
                ref_atoms_sup.append(r_atom)
                des_atoms_sup.append(d_atom)
                shared_ids.append(res_id)
            except KeyError:
                continue

        if len(ref_atoms_sup) < 3:
            return 99.0

        # Superpose designed chain A onto reference chain A
        sup = Superimposer()
        sup.set_atoms(ref_atoms_sup, des_atoms_sup)
        sup.apply(des_struct.get_atoms())   # transforms entire designed structure

        # Now compute RMSD on fixed motif residues only
        sq_devs = []
        for res_id in FIXED_MOTIF:
            try:
                ref_ca = ref_struct[0]['A'][res_id]['CA'].get_vector().get_array()
                des_ca = des_struct[0]['A'][res_id]['CA'].get_vector().get_array()
                diff   = des_ca - ref_ca
                sq_devs.append(np.dot(diff, diff))
            except KeyError:
                continue

        return float(np.sqrt(np.mean(sq_devs))) if sq_devs else 99.0

    except Exception as e:
        print(f'  Error on {os.path.basename(pdb_path)}: {e}')
        return 99.0

# ── Parse all 100 designs ──────────────────────────────────────────────────────
results = []

for pdb_path in output_pdbs:
    design_id = os.path.basename(pdb_path).replace('.pdb', '')
    trb_path  = pdb_path.replace('.pdb', '.trb')
    entry     = {'id': design_id, 'pdb': pdb_path}

    if os.path.exists(trb_path):
        trb = np.load(trb_path, allow_pickle=True)
        if isinstance(trb, np.ndarray):
            trb = trb.item()
        plddt_arr      = trb.get('plddt', np.array([[0]]))
        final_plddt    = plddt_arr[-1] if plddt_arr.ndim == 2 else plddt_arr
        entry['plddt'] = float(np.mean(final_plddt))
        entry['length'] = int(plddt_arr.shape[-1])
    else:
        entry['plddt']  = 0.0
        entry['length'] = 0

    entry['motif_rmsd'] = compute_motif_rmsd(pdb_path)
    results.append(entry)

# ── Score distribution ─────────────────────────────────────────────────────────
plddt_vals = [r['plddt'] for r in results]
rmsd_vals  = [r['motif_rmsd'] for r in results if r['motif_rmsd'] < 90]
lens       = [r['length'] for r in results]

print(f'\nScore distribution:')
print(f'  pLDDT range:      {min(plddt_vals):.3f} — {max(plddt_vals):.3f}')
print(f'  pLDDT mean:       {np.mean(plddt_vals):.3f}')
if rmsd_vals:
    print(f'  Motif RMSD range: {min(rmsd_vals):.4f} — {max(rmsd_vals):.4f} Å')
    print(f'  Motif RMSD mean:  {np.mean(rmsd_vals):.4f} Å')
print(f'  Length range:     {min(lens)} — {max(lens)} residues')

# ── Kill switch filter ─────────────────────────────────────────────────────────
PLDDT_MIN = 0.80
RMSD_MAX  = 1.0

passed = sorted(
    [r for r in results if r['plddt'] >= PLDDT_MIN
                        and r['motif_rmsd'] <= RMSD_MAX],
    key=lambda x: (x['motif_rmsd'], -x['plddt'])
)

print(f'\nFilter: pLDDT >= {PLDDT_MIN} AND motif RMSD <= {RMSD_MAX} Å')
print(f'  Passed: {len(passed)} / {len(results)}')

if len(passed) > 0:
    print(f'\nTop 10 designs:')
    print(f'{"Design":<30} {"pLDDT":>8} {"RMSD (Å)":>10} {"Length":>8}')
    print('-' * 58)
    for r in passed[:10]:
        print(f'{r["id"]:<30} {r["plddt"]:>8.4f} '
              f'{r["motif_rmsd"]:>10.4f} {r["length"]:>8}')

Copied 200 files in 1.5s
PDB files: 100
TRB files: 100
Designs generated:  100
Score files (.trb): 100

Score distribution:
  pLDDT range:      0.993 — 0.997
  pLDDT mean:       0.995
  Motif RMSD range: 0.0537 — 0.1979 Å
  Motif RMSD mean:  0.1224 Å
  Length range:     115 — 165 residues

Filter: pLDDT >= 0.8 AND motif RMSD <= 1.0 Å
  Passed: 100 / 100

Top 10 designs:
Design                            pLDDT   RMSD (Å)   Length
----------------------------------------------------------
myc_binder_88                    0.9961     0.0537      132
myc_binder_66                    0.9943     0.0774      126
myc_binder_75                    0.9951     0.0801      142
myc_binder_44                    0.9944     0.0809      118
myc_binder_74                    0.9959     0.0828      146
myc_binder_46                    0.9940     0.0879      158
myc_binder_79                    0.9953     0.0891      120
myc_binder_22                    0.9953     0.0895      133
myc_binder_96               

In [8]:
# ── Cell 10: Save ALL passing candidates and push to Drive ─────────────────────
import json, os, shutil

# Pass all 100 — kill switches downstream will filter naturally
TOP_N          = len(passed)   # all 100
MPNN_INPUT_DIR = f'{DRIVE_DIR}/proteinmpnn_inputs'
RFDIFF_DRIVE   = f'{DRIVE_DIR}/rfdiffusion_outputs'

os.makedirs(MPNN_INPUT_DIR, exist_ok=True)
os.makedirs(RFDIFF_DRIVE,   exist_ok=True)

top_candidates = passed   # all 100 sorted by RMSD then pLDDT

print(f'Forwarding all {TOP_N} passing candidates to ProteinMPNN:')
for i, c in enumerate(top_candidates):
    dst = f'{MPNN_INPUT_DIR}/candidate_{i+1}_{c["id"]}.pdb'
    shutil.copy2(c['pdb'], dst)

print(f'  Saved {TOP_N} PDB files to {MPNN_INPUT_DIR}')

# Push all RFdiffusion outputs to Drive
print(f'\nPushing all RFdiffusion outputs to Drive...')
pushed = 0
for fname in os.listdir(RFDIFF_OUT):
    shutil.copy2(
        f'{RFDIFF_OUT}/{fname}',
        f'{RFDIFF_DRIVE}/{fname}'
    )
    pushed += 1
print(f'  {pushed} files pushed to Drive')

# Update pipeline config
with open(f'{DRIVE_DIR}/pipeline_config.json') as f:
    cfg = json.load(f)

cfg['rfdiff_designs_generated'] = len(results)
cfg['rfdiff_designs_passed']    = len(passed)
cfg['rfdiff_top_candidates']    = [c['id'] for c in top_candidates]
cfg['rfdiff_top_plddt']         = [c['plddt'] for c in top_candidates]
cfg['rfdiff_top_rmsd']          = [c['motif_rmsd'] for c in top_candidates]
cfg['proteinmpnn_input_dir']    = MPNN_INPUT_DIR

with open(f'{DRIVE_DIR}/pipeline_config.json', 'w') as f:
    json.dump(cfg, f, indent=2)

print(f'\nPipeline config updated.')
print(f'\nPhase II complete.')
print(f'  Generated:   {len(results)} backbones')
print(f'  Passed:      {len(passed)} quality filter')
print(f'  Forwarded:   {TOP_N} to ProteinMPNN')
print(f'\nExpected attrition through kill switches:')
print(f'  After ProteinMPNN + ESMFold:     ~60-80 remaining')
print(f'  After OpenMM MD stability:       ~30-50 remaining')
print(f'  After MM-GBSA binding energy:    ~15-25 remaining')
print(f'  After membrane PMF:              ~5-10 remaining')
print(f'  After ternary complex sim:       ~3-6 remaining')
print(f'  Candidates submitted for synthesis: 3-6')
print(f'\nOpen ProteinMPNN notebook next.')

Forwarding all 100 passing candidates to ProteinMPNN:
  Saved 100 PDB files to /content/drive/MyDrive/myc_gnn/proteinmpnn_inputs

Pushing all RFdiffusion outputs to Drive...
  200 files pushed to Drive

Pipeline config updated.

Phase II complete.
  Generated:   100 backbones
  Passed:      100 quality filter
  Forwarded:   100 to ProteinMPNN

Expected attrition through kill switches:
  After ProteinMPNN + ESMFold:     ~60-80 remaining
  After OpenMM MD stability:       ~30-50 remaining
  After MM-GBSA binding energy:    ~15-25 remaining
  After membrane PMF:              ~5-10 remaining
  After ternary complex sim:       ~3-6 remaining
  Candidates submitted for synthesis: 3-6

Open ProteinMPNN notebook next.
